In [2]:
from pathlib import Path

import pymupdf
from PIL import Image, ImageDraw
import ipywidgets as widgets
from IPython.display import display, clear_output


# ---------- Open PDF ----------
PROJECT_ROOT = Path.cwd().parent
PDF_PATH = PROJECT_ROOT / "samples" / "test.pdf"

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find PDF: {PDF_PATH}\n"
        "Check the filename and samples folder."
    )

document = pymupdf.open(PDF_PATH)


# ---------- Reader state ----------
reader_state = {
    "current_page": 0,
    "zoom_dpi": 120,
    "search_results": [],
    "search_index": 0,
    "bookmarks": [],
}


# ---------- Render PDF page ----------
def render_page(page_number, dpi, highlight_rectangles=None):
    page = document[page_number]
    pixmap = page.get_pixmap(dpi=dpi)

    mode = "RGBA" if pixmap.alpha else "RGB"

    image = Image.frombytes(
        mode,
        (pixmap.width, pixmap.height),
        pixmap.samples
    )

    if highlight_rectangles:
        draw = ImageDraw.Draw(image, "RGBA")
        scale = dpi / 72

        for rect in highlight_rectangles:
            draw.rectangle(
                [
                    rect.x0 * scale,
                    rect.y0 * scale,
                    rect.x1 * scale,
                    rect.y1 * scale
                ],
                fill=(255, 235, 0, 90),
                outline=(255, 0, 0, 255),
                width=3
            )

    return image


# ---------- Create interface ----------
previous_button = widgets.Button(description="◀ Previous")
next_button = widgets.Button(description="Next ▶")

zoom_out_button = widgets.Button(description="− Zoom")
zoom_in_button = widgets.Button(description="+ Zoom")

page_input = widgets.BoundedIntText(
    value=1,
    min=1,
    max=document.page_count,
    description="Page:"
)

go_button = widgets.Button(description="Go")

page_label = widgets.Label()
zoom_label = widgets.Label()

search_input = widgets.Text(
    placeholder="Type a word or phrase...",
    description="Search:"
)

search_button = widgets.Button(
    description="Search",
    button_style="info"
)

previous_result_button = widgets.Button(
    description="◀ Previous result"
)

next_result_button = widgets.Button(
    description="Next result ▶"
)

search_status = widgets.Label(value="Search is ready.")

page_output = widgets.Output()

navigation_bar = widgets.HBox([
    previous_button,
    next_button,
    zoom_out_button,
    zoom_in_button,
    page_input,
    go_button,
    page_label,
    zoom_label
])

search_bar = widgets.HBox([
    search_input,
    search_button,
    previous_result_button,
    next_result_button,
    search_status
])


# ---------- Refresh the displayed page ----------
def refresh_reader():
    current_page = reader_state["current_page"]
    dpi = reader_state["zoom_dpi"]

    page_input.value = current_page + 1
    page_label.value = f"Page {current_page + 1} / {document.page_count}"
    zoom_label.value = f"Render: {dpi} DPI"

    previous_button.disabled = (current_page == 0)
    next_button.disabled = (current_page == document.page_count - 1)

    highlights = None
    results = reader_state["search_results"]

    if results:
        current_result = results[reader_state["search_index"]]

        if current_result["page_number"] == current_page:
            highlights = current_result["rectangles"]

    with page_output:
        clear_output(wait=True)
        display(render_page(current_page, dpi, highlights))


# ---------- Page navigation ----------
def show_previous_page(button):
    if reader_state["current_page"] > 0:
        reader_state["current_page"] -= 1
        refresh_reader()


def show_next_page(button):
    if reader_state["current_page"] < document.page_count - 1:
        reader_state["current_page"] += 1
        refresh_reader()


def go_to_page(button):
    reader_state["current_page"] = page_input.value - 1
    refresh_reader()


def zoom_out(button):
    if reader_state["zoom_dpi"] > 72:
        reader_state["zoom_dpi"] -= 24
        refresh_reader()


def zoom_in(button):
    if reader_state["zoom_dpi"] < 240:
        reader_state["zoom_dpi"] += 24
        refresh_reader()


# ---------- Search ----------
def search_document(button=None):
    term = search_input.value.strip()

    if not term:
        search_status.value = "Enter text to search."
        return

    results = []

    for page_number, page in enumerate(document):
        rectangles = page.search_for(term)

        if rectangles:
            results.append({
                "page_number": page_number,
                "rectangles": rectangles
            })

    reader_state["search_results"] = results
    reader_state["search_index"] = 0

    if not results:
        search_status.value = f'No results for "{term}".'
        refresh_reader()
        return

    show_current_result()


def show_current_result():
    result = reader_state["search_results"][reader_state["search_index"]]

    reader_state["current_page"] = result["page_number"]

    search_status.value = (
        f'Result {reader_state["search_index"] + 1} / '
        f'{len(reader_state["search_results"])} '
        f'on page {result["page_number"] + 1}'
    )

    refresh_reader()


def show_next_result(button):
    if not reader_state["search_results"]:
        search_status.value = "Search for a word first."
        return

    reader_state["search_index"] = (
        reader_state["search_index"] + 1
    ) % len(reader_state["search_results"])

    show_current_result()


def show_previous_result(button):
    if not reader_state["search_results"]:
        search_status.value = "Search for a word first."
        return

    reader_state["search_index"] = (
        reader_state["search_index"] - 1
    ) % len(reader_state["search_results"])

    show_current_result()


# ---------- Connect button events ----------

previous_button.on_click(show_previous_page)
next_button.on_click(show_next_page)
go_button.on_click(go_to_page)

zoom_out_button.on_click(zoom_out)
zoom_in_button.on_click(zoom_in)

search_button.on_click(search_document)
next_result_button.on_click(show_next_result)
previous_result_button.on_click(show_previous_result)


# ---------- Display complete reader ----------
display(navigation_bar)
display(search_bar)
display(page_output)

refresh_reader()

#----------Add Bookmarks ------------------

# bookmark-controls cell
bookmark_label_input = widgets.Text(
    placeholder="Example: Introduction or Important equation",
    description="Label:"
)

add_bookmark_button = widgets.Button(
    description="☆ Add bookmark",
    button_style="success"
)

bookmark_status = widgets.Label(
    value="No bookmarks yet."
)

bookmarks_output = widgets.Output()

bookmark_bar = widgets.HBox([
    bookmark_label_input,
    add_bookmark_button,
    bookmark_status
])

display(bookmark_bar)
display(bookmarks_output)

def refresh_bookmarks():
    with bookmarks_output:
        clear_output(wait=True)

        bookmarks = reader_state["bookmarks"]

        if not bookmarks:
            print("No bookmarks saved.")
            return

        print("Saved bookmarks:")

        for index, bookmark in enumerate(bookmarks):
            page_number = bookmark["page_number"]
            label = bookmark["label"]

            open_button = widgets.Button(
                description=f"Page {page_number + 1}: {label}",
                layout=widgets.Layout(width="400px")
            )

            delete_button = widgets.Button(
                description="Delete",
                button_style="danger",
                layout=widgets.Layout(width="90px")
            )

            def open_bookmark(button, target_page=page_number):
                reader_state["current_page"] = target_page
                refresh_reader()

            def delete_bookmark(button, bookmark_index=index):
                reader_state["bookmarks"].pop(bookmark_index)
                refresh_bookmarks()

            open_button.on_click(open_bookmark)
            delete_button.on_click(delete_bookmark)

            display(widgets.HBox([open_button, delete_button]))

#  bookmark-logic cell
def add_bookmark(button):
    current_page = reader_state["current_page"]
    label = bookmark_label_input.value.strip()

    if not label:
        label = f"Page {current_page + 1}"

    already_exists = any(
        bookmark["page_number"] == current_page
        and bookmark["label"] == label
        for bookmark in reader_state["bookmarks"]
    )

    if already_exists:
        bookmark_status.value = "That bookmark already exists."
        return

    reader_state["bookmarks"].append({
        "page_number": current_page,
        "label": label
    })

    bookmark_label_input.value = ""
    bookmark_status.value = (
        f'Bookmark saved for page {current_page + 1}.'
    )

    refresh_bookmarks()


add_bookmark_button.on_click(add_bookmark)

refresh_bookmarks()

Output()

Output()